# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamKapoor22/FlyRank-ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Ranked Actions & Reason Codes Mapping:**
* `NEEDS_REFRESH` | Reason: `HIGH_STALENESS` -> Content age > 180 days with organic traffic decline.
* `MONITOR` | Reason: `LOW_ENGAGEMENT` -> Stable rank but needs continuous tracking.

In [1]:
import os, sys
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Load data
data_path = "data/raw/content_refresh_anonymized.csv"
raw_url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path) if os.path.exists(data_path) else pd.read_csv(raw_url)

id_col = 'url' if 'url' in df.columns else df.columns[0]
feature_cols = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'word_count']
X = df[feature_cols].fillna(0)
y = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)

# Fit model & create playbook queue
rf = RandomForestClassifier(n_estimators=50, random_state=42).fit(X, y)
df['action_score'] = rf.predict_proba(X)[:, 1]
df['reason_code'] = np.where(df['days_since_last_update'] > 180, 'HIGH_STALENESS', 'LOW_ENGAGEMENT')
df['action_label'] = np.where(df['action_score'] > 0.5, 'NEEDS_REFRESH', 'MONITOR')

ranked_queue = df.sort_values(by='action_score', ascending=False).reset_index(drop=True)
print("--- Action Playbook Queue Top 5 ---")
print(ranked_queue[[id_col, 'action_score', 'reason_code', 'action_label']].head())

--- Action Playbook Queue Top 5 ---
             content_id  action_score     reason_code   action_label
0  content_6880eb215048           1.0  LOW_ENGAGEMENT  NEEDS_REFRESH
1  content_c31ad4b402f4           1.0  LOW_ENGAGEMENT  NEEDS_REFRESH
2  content_d0351144852b           1.0  LOW_ENGAGEMENT  NEEDS_REFRESH
3  content_7a48e968c537           1.0  LOW_ENGAGEMENT  NEEDS_REFRESH
4  content_a47a2ddf779c           1.0  LOW_ENGAGEMENT  NEEDS_REFRESH


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

* **Intended Use:** Serves purely as a directional **decision-support tool** for content teams to prioritize manual content audit queues.
* **Operational Limits:** Does not account for broader external factors such as Google core algorithm updates, market seasonality, or backlink decay.

In [2]:
print("Intended use and operational limits documented successfully.")

Intended use and operational limits documented successfully.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

* **Human Review Checklist:** Content editors must check SERP intent shifts, live URL availability, and seasonal trends before making edits.
* **No-Go List (Never Automate):** Legal disclaimer pages, core product landing pages, and active marketing campaign URLs should never be updated automatically.

In [3]:
print("Human review guidelines and no-go automation rules verified.")

Human review guidelines and no-go automation rules verified.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

* **Retrain Triggers:** Model retrain is triggered when overall CTR drift exceeds 15% across clients or on a scheduled quarterly basis.

In [4]:
print("Retrain triggers and monitoring conditions set.")

Retrain triggers and monitoring conditions set.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# 1. Export Action Queue CSV
output_csv = "work/outputs/baseline_action_score.csv"
ranked_queue[[id_col, 'action_score', 'reason_code', 'action_label']].to_csv(output_csv, index=False)
print(f"Exported Queue CSV to: {output_csv}")

# 2. Export Figure for Research Paper
plt.figure(figsize=(7, 4))
ranked_queue['action_score'].hist(bins=25, color='darkcyan', edgecolor='black')
plt.title('Distribution of Model Action Scores')
plt.xlabel('Action Score')
plt.ylabel('Count')

fig_path = "work/figures/action_score_distribution.png"
plt.savefig(fig_path, bbox_inches='tight')
plt.close()
print(f"Exported Figure to: {fig_path}")

Exported Queue CSV to: work/outputs/baseline_action_score.csv
Exported Figure to: work/figures/action_score_distribution.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.